# T-AIA-911 — Travel Order Resolver

**Démonstration du pipeline complet** : NER → Pathfinding → Horaires

Ce notebook montre le fonctionnement de chaque composant du projet final.

In [ ]:
import sys
from pathlib import Path

# Ajouter final_project au path
# Le notebook est exécuté depuis final_project/, donc on remonte d'un niveau
PROJECT_ROOT = Path(".").resolve().parent
FINAL = PROJECT_ROOT / "final_project"
if str(FINAL) not in sys.path:
    sys.path.insert(0, str(FINAL))

print(f"Project root : {PROJECT_ROOT}")
print(f"Final project: {FINAL}")

Project root : /Users/atressel/Documents/Epitech/Projets/T-AIA-911-NAN_1/final_project
Final project: /Users/atressel/Documents/Epitech/Projets/T-AIA-911-NAN_1/final_project/final_project


## 1. Chargement du modèle NER

Le modèle `PikPak44/yann_best_model` est téléchargé depuis Hugging Face Hub (avec cache local).
Il utilise des labels **ORIGIN** / **DEST** pour distinguer départ et arrivée.

In [2]:
from ner import download_hf_model, load_ner

HF_MODEL = "PikPak44/yann_best_model"

# Téléchargement (cache automatique dans ~/.cache/huggingface/hub/)
model_path = download_hf_model(HF_MODEL)
print(f"Modèle local : {model_path}")

# Chargement du pipeline NER
ner_bundle = load_ner(model_path)
print(f"Mode DEP/ARR : {ner_bundle['use_dep_arr']}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Modèle local : /Users/atressel/.cache/huggingface/hub/models--PikPak44--yann_best_model/snapshots/d5725db4a46ce181970d58b9d8f5b3fa8acedf97


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Mode DEP/ARR : True


## 2. Extraction d'entités NER

Test sur plusieurs phrases représentatives du sujet.

In [3]:
from ner import extract_cities

test_phrases = [
    "Je veux aller de Nantes à Paris",
    "Je souhaite me rendre à Lyon depuis Marseille",
    "A quelle heure y a-t-il des trains vers Bordeaux en partance de Toulouse ?",
    "Avec mes amis florence et paris, je voudrais aller de paris a florence.",
    "Comment me rendre à Port Boulet depuis Tours ?",
    "Je veux aller de toulouse a paris en passant par limoges",
    "Je voudrais un billet Rennes Strasbourg",
    "Je veux la recette du paris-brest",
]

for phrase in test_phrases:
    cities = extract_cities(phrase, ner_bundle)
    status = "✅" if len(cities) >= 2 else ("⚠️" if cities else "❌")
    print(f"{status} \"{phrase}\"")
    print(f"   → {cities}")
    print()

✅ "Je veux aller de Nantes à Paris"
   → ['Nantes', 'Paris']

✅ "Je souhaite me rendre à Lyon depuis Marseille"
   → ['Marseille', 'Lyon']

✅ "A quelle heure y a-t-il des trains vers Bordeaux en partance de Toulouse ?"
   → ['Toulouse', 'Bordeaux']

✅ "Avec mes amis florence et paris, je voudrais aller de paris a florence."
   → ['paris', 'florence']

✅ "Comment me rendre à Port Boulet depuis Tours ?"
   → ['Tours', 'Port Boulet']

✅ "Je veux aller de toulouse a paris en passant par limoges"
   → ['toulouse', 'paris']

✅ "Je voudrais un billet Rennes Strasbourg"
   → ['Rennes', 'Strasbourg']

✅ "Je veux la recette du paris-brest"
   → ['paris', '-brest']



## 3. Construction du graphe GTFS

Le `SNCFPathFinder` charge les fichiers GTFS (stops, stop_times, trips) et construit :
- Un **graphe pondéré** (poids = temps de trajet minimum entre gares)
- Un **index des horaires** par trip, par arête et par gare

In [4]:
from pathfinder import SNCFPathFinder

GTFS_DIR = str(PROJECT_ROOT / "dataset" / "sncf")
GEO_CSV = str(PROJECT_ROOT / "dataset" / "geo.csv")

pf = SNCFPathFinder(gtfs_dir=GTFS_DIR, geo_csv_path=GEO_CSV)
pf.build()

print(f"Gares          : {len(pf.area_id_to_name)}")
print(f"Gares connectées: {len(pf.graph)}")
print(f"Arêtes         : {sum(len(v) for v in pf.graph.values())}")
print(f"Trips indexés  : {len(pf._trip_schedule)}")

FileNotFoundError: Fichier GTFS manquant : /Users/atressel/Documents/Epitech/Projets/T-AIA-911-NAN_1/final_project/dataset/sncf/stops.txt

## 4. Pathfinding — Dijkstra

Recherche du plus court chemin entre deux villes.

In [ ]:
from utils import format_duration

route = pf.route_with_waypoints(["Nantes", "Paris"])

print(f"Départ  : {route.matched_stations[0]}")
print(f"Arrivée : {route.matched_stations[-1]}")
print(f"Temps   : {format_duration(route.total_time_min)}")
print(f"Gares   : {len(route.station_names)}")
print()
print("Itinéraire :")
for i, name in enumerate(route.station_names):
    prefix = "🚉" if i == 0 or i == len(route.station_names) - 1 else "  •"
    print(f"  {prefix} {name}")

## 5. Itinéraire multi-étapes (waypoints)

Test avec une étape intermédiaire : Toulouse → Paris en passant par Limoges.

In [ ]:
route_wp = pf.route_with_waypoints(["Toulouse", "Limoges", "Paris"])

print(f"Villes demandées : {route_wp.requested_cities}")
print(f"Gares matchées   : {route_wp.matched_stations}")
print(f"Temps total      : {format_duration(route_wp.total_time_min)}")
print()
for seg in route_wp.segments:
    print(f"  {seg.from_station_name} → {seg.to_station_name} ({format_duration(seg.time_min)}, {seg.train_count} trains/jour)")

## 6. Horaires concrets & correspondances

Le `find_schedule_for_route` trouve des trains réels pour un itinéraire, en maximisant la couverture (un train qui couvre le plus de gares possible = moins de correspondances).

In [ ]:
schedule = pf.find_schedule_for_route(route.station_ids)

if schedule:
    print(f"Nombre de trains : {len(schedule.legs)}")
    print(f"Temps total      : {format_duration(schedule.total_time_min)}")
    if schedule.total_wait_min > 0:
        print(f"Attente totale   : {format_duration(schedule.total_wait_min)}")
    print()

    for i, leg in enumerate(schedule.legs):
        if i > 0 and i - 1 < len(schedule.connections):
            conn = schedule.connections[i - 1]
            print(f"  🔄 Correspondance à {conn.station_name} — {format_duration(conn.wait_min)} d'attente ({conn.arrive_time} → {conn.depart_time})")
            print()

        t = leg.train_type or "Train"
        print(f"  🚆 {t} n°{leg.train_number}")
        for j, stop in enumerate(leg.stops):
            if j == 0:
                print(f"     {stop.departure_time}  {stop.station_name} (départ)")
            elif j == len(leg.stops) - 1:
                print(f"     {stop.arrival_time}  {stop.station_name} (arrivée)")
            else:
                print(f"     {stop.arrival_time}  {stop.station_name}")
        print()
else:
    print("Aucun horaire trouvé.")

## 7. Pipeline complet (NER → Route → Horaires)

Simulation de ce que fait l'UI Streamlit : phrase utilisateur → extraction → pathfinding → horaires.

In [ ]:
def pipeline_demo(phrase: str):
    """Pipeline complet : phrase → NER → route → horaires."""
    print(f"📝 \"{phrase}\"")
    print()

    # 1. NER
    cities = extract_cities(phrase, ner_bundle)
    if len(cities) < 2:
        print(f"   ❌ Pas assez de villes détectées : {cities}")
        return
    print(f"   🏙️ Villes : {cities}")

    # 2. Pathfinding
    try:
        route = pf.route_with_waypoints(cities)
    except Exception as e:
        print(f"   ❌ Erreur pathfinding : {e}")
        return
    print(f"   🗺️ {route.matched_stations[0]} → {route.matched_stations[-1]} ({format_duration(route.total_time_min)})")

    # 3. Horaires
    schedule = pf.find_schedule_for_route(route.station_ids)
    if schedule and schedule.legs:
        trains = [f"{leg.train_type} n°{leg.train_number}" for leg in schedule.legs]
        print(f"   🚆 Trains : {' → '.join(trains)}")
        if schedule.connections:
            for c in schedule.connections:
                print(f"   🔄 Correspondance à {c.station_name} ({format_duration(c.wait_min)})")
    print()


# Tests
pipeline_demo("Je veux aller de Nantes à Marseille")
pipeline_demo("Je souhaite me rendre à Strasbourg depuis Bordeaux")
pipeline_demo("Je veux aller de Hennebont à Paris")
pipeline_demo("Je veux la recette du paris-brest")

## 8. Évaluation NER sur fichier (format sujet)

Test du format d'I/O demandé par le sujet : `sentenceID,sentence` → `sentenceID,Departure,Destination`

In [ ]:
eval_sentences = [
    "Je veux aller de Paris à Lyon",
    "Comment me rendre à Bordeaux depuis Nantes ?",
    "Bonjour, quel temps fait-il ?",
    "Je voudrais un billet Toulouse Marseille",
    "Je veux la recette du paris-brest",
    "Avec mes amis florence et paris, je voudrais aller de paris a florence.",
]

print("sentenceID,Departure,Destination")
print("-" * 50)
for idx, sentence in enumerate(eval_sentences, 1):
    cities = extract_cities(sentence, ner_bundle)
    if len(cities) >= 2:
        dep, arr = cities[0], cities[-1]
        steps = cities[1:-1]
        row = [str(idx), dep] + steps + [arr]
        print(",".join(row))
    else:
        print(f"{idx},INVALID,")

## 9. Statistiques du graphe SNCF

In [ ]:
import pandas as pd

# Top 10 gares les plus connectées
degree_data = [
    {"Gare": pf.area_id_to_name.get(sid, sid), "Connexions": deg}
    for sid, deg in sorted(pf.station_degree.items(), key=lambda x: x[1], reverse=True)[:10]
]
df_top = pd.DataFrame(degree_data)
print("Top 10 gares les plus connectées :")
print(df_top.to_string(index=False))
print()

# Distribution des temps de trajet
times = list(pf.edge_time_by_pair.values())
print(f"Nombre d'arêtes : {len(times)}")
print(f"Temps min       : {format_duration(min(times))}")
print(f"Temps max       : {format_duration(max(times))}")
print(f"Temps moyen     : {format_duration(sum(times) / len(times))}")

---

**Résumé** : Ce notebook démontre le pipeline complet du projet T-AIA-911 :
1. **NER** — Extraction de villes avec modèle fine-tuné (ORIGIN/DEST)
2. **Pathfinding** — Dijkstra sur graphe GTFS SNCF
3. **Horaires** — Trains concrets avec correspondances et temps d'attente
4. **Format sujet** — I/O `sentenceID,Departure,Destination`